# Baseline Modeling and Error Analysis

The objective of this notebook is to:
- Build baseline models,
- Compare their results,
- Analyze false negatives,
- Analyze false positives,
- Formulate hypotheses for further experiments.

Import bibliotek

In [1]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

Funkcje

In [2]:
DECISION_THRESHOLD = 0.5

def evaluate_model(name, model, X_test, y_test, threshold=DECISION_THRESHOLD):
    y_pred = (y.proba >= threshold).astype(int)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(name)

    print("Decision Threshold:", threshold)
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, y_pred), 4))
    print("F1:", round(f1_score(y_test, y_pred), 4))
    print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
    print("PR-AUC:", round(average_precision_score(y_test, y_proba), 4))

    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    
    print("Classification Report:\n", classification_report(y_test, y_pred, zero_division=0))


def get_metrics(name, model, X_test, y_test, threshold=DECISION_THRESHOLD):
    y_pred = (y_proba >= threshold).astype(int)
    y_proba = model.predict_proba(X_test)[:, 1]

    return {
        "model": name,
        "decision_threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba)
    }

def get_false_negatives(model):
    return debug_df[(debug_df['y_true']==1) & (debug_df[f'{model}_pred']== 0)]

Wczytanie danych

In [3]:
X_train = pd.read_csv('../data/processed/X_train_scaled.csv')
X_test = pd.read_csv('../data/processed/X_test_scaled.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

Test wczytania danych

In [4]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

print(X_train.head())
print(y_train.value_counts())
print(y_test.value_counts())

(8000, 8)
(2000, 8)
(8000,)
(2000,)
   Type_L  Type_M  Process temperature [K]  Temperature difference  \
0       1       0                 0.874402               -0.800498   
1       1       0                -0.408876                1.900381   
2       0       1                 0.536697               -0.500400   
3       0       1                -0.071171                1.100121   
4       1       0                 0.198993               -0.900531   

   Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  
0               -0.100148    -0.851496  -1.182391         1.485820  
1               -0.122496    -0.329786  -0.421646         1.973739  
2               -0.167192     0.011332   0.052397        -1.300041  
3               -0.692367     1.325640   1.499616         1.359905  
4               -0.223061    -0.189326  -0.280724        -0.733425  
Machine failure
0    7736
1     264
Name: count, dtype: int64
Machine failure
0    1934
1      66
Name: count, dtype: int64


## Tworzenie Modeli

Tworzenie modelu naiwnego

In [5]:
dummy = DummyClassifier(strategy='most_frequent')

dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_proba_dummy = dummy.predict_proba(X_test)[:, 1]

In [6]:
evaluate_model('Dummy Classifier', dummy, X_test, y_test)

NameError: name 'y' is not defined

Wnioski: Model naiwny ma bardzo wysoką Accuracy ale nie przewidział żadnego Machine failure. Model jest bezużyteczny



Logistic Regression

In [ ]:
log_reg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

log_reg.fit(X_train, y_train)

y_pred_log = log_reg.predict(X_test)
y_proba_log = log_reg.predict_proba(X_test)[:, 1]


In [ ]:
evaluate_model('LogisticRegression', log_reg, X_test, y_test)

Wnioski: Model Logistic Regression osiągnął mniejsze Accuracy ale udało mu się uzyskać Precision na poziomie 16% - poradził sobie lepiej od modelu naiwnego ale jest nieoptymalny do tego zadania.



Random Forest

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

random_forest.fit(X_train, y_train)

evaluate_model('RandomForestClassifier', random_forest, X_test, y_test)

Wnioski: Model Random Forest poradził sobie bardzo dobrze z wynikiem precyzji równym 86%. Nie udało mu się przewidzieć 11 rekordów oraz dał fałszywy alarm dla 9 rekordów.

Gradient Boosting


In [ ]:
gradient_boosting = GradientBoostingClassifier(
    random_state=42
)

gradient_boosting.fit(X_train, y_train)

evaluate_model('GradientBoostingClassifier', gradient_boosting, X_test, y_test)

Wnioski: Model Gradient Boosting poradził sobie bardzo dobrze - osiągnął wynik precyzji 88%. Natomiast nie udało mu się przewidzieć 14 machine failure oraz dał fałszywy alarm dla 7 rekordów. Zmalała liczba fałszywych alarmów ale wzrosła liczba nieprzewidzianych awarii.

Porównanie modeli

In [ ]:
results = []

results.append(get_metrics("Dummy", dummy, X_test, y_test))
results.append(get_metrics("Logistic Regression", log_reg, X_test, y_test))
results.append(get_metrics("Random Forest", random_forest, X_test, y_test))
results.append(get_metrics("Gradient Boosting", gradient_boosting, X_test, y_test))

results_df = pd.DataFrame(results)
print(results_df.round(4))

In [ ]:
#results_df.to_csv("../data/processed/model_results_baseline.csv", index=False)

## Analiza przeoczonych awarii - False Negatives

In [ ]:
from sklearn.model_selection import train_test_split

df_clean = pd.read_csv('../data/processed/produkcja_clean.csv')

X_raw = df_clean.drop(columns=['Machine failure'])
y_raw = df_clean['Machine failure']

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw,
    y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw
)

X_test_raw = X_test_raw.copy()

In [ ]:
print("X_test scaled shape:", X_test.shape)
print("X_test raw shape:", X_test_raw.shape)

print("y_test shape:", y_test.shape)
print("y_test_raw shape:", y_test_raw.shape)

In [ ]:
print((y_test.reset_index(drop=True) == y_test_raw.reset_index(drop=True)).all())

In [ ]:
debug_df = X_test_raw.copy()
debug_df['y_true'] = y_test_raw.values

models = {
    'log_reg': log_reg,
    'random_forest': random_forest,
    'gradient_boosting': gradient_boosting
}

for name, model in models.items():
    debug_df[f"{name}_pred"] = model.predict(X_test)
    debug_df[f"{name}_proba"] = model.predict_proba(X_test)[:, 1]

print(debug_df.head())
print(debug_df.shape)

In [ ]:
debug_df[[
    "y_true",
    "log_reg_pred",
    "log_reg_proba",
    "random_forest_pred",
    "random_forest_proba",
    "gradient_boosting_pred",
    "gradient_boosting_proba"
]].head(10)

In [ ]:
get_false_negatives('log_reg')

In [ ]:
print(get_false_negatives('random_forest'))

In [ ]:
print(get_false_negatives('gradient_boosting'))

In [ ]:
fn_summary = pd.DataFrame({
    'model': ['log_reg', 'random_forest', 'gradient_boosting'],
    'false_negatives': [get_false_negatives('log_reg').shape[0],
                        get_false_negatives('random_forest').shape[0],
                        get_false_negatives('gradient_boosting').shape[0]]
})
print(fn_summary)

In [ ]:
fn_log_index = set(get_false_negatives('log_reg').index)
fn_rf_index = set(get_false_negatives('random_forest').index)
fn_gb_index = set(get_false_negatives('gradient_boosting').index)

print("Logistic Regression FN:", len(fn_log_index))
print("Random Forest FN:", len(fn_rf_index))
print("Gradient Boosting FN:", len(fn_gb_index))

In [ ]:
fn_all_index = fn_log_index & fn_rf_index & fn_gb_index

print("Awarie przeoczone przez wszystkie modele:", len(fn_all_index))
print(fn_all_index)

In [ ]:
fn_all = debug_df.loc[list(fn_all_index)].copy()
print(fn_all)

In [ ]:
overlap_summary = pd.DataFrame({
    "comparison": [
        "Logistic Regression ∩ Random Forest",
        "Logistic Regression ∩ Gradient Boosting",
        "Random Forest ∩ Gradient Boosting",
        "All three models"
    ],
    "shared_false_negatives": [
        len(fn_log_index & fn_rf_index),
        len(fn_log_index & fn_gb_index),
        len(fn_rf_index & fn_gb_index),
        len(fn_all_index)
    ]
})

print(overlap_summary)

In [ ]:
debug_df["missed_by_log_reg"] = (
    (debug_df["y_true"] == 1) &
    (debug_df["log_reg_pred"] == 0)
)

debug_df["missed_by_random_forest"] = (
    (debug_df["y_true"] == 1) &
    (debug_df["random_forest_pred"] == 0)
)

debug_df["missed_by_gradient_boosting"] = (
    (debug_df["y_true"] == 1) &
    (debug_df["gradient_boosting_pred"] == 0)
)

debug_df["missed_by_n_models"] = (
    debug_df["missed_by_log_reg"].astype(int) +
    debug_df["missed_by_random_forest"].astype(int) +
    debug_df["missed_by_gradient_boosting"].astype(int)
)

In [ ]:
missed_count_summary = (
    debug_df[debug_df["y_true"] == 1]["missed_by_n_models"]
    .value_counts()
    .sort_index()
)

print(missed_count_summary)

In [ ]:
hard_cases = debug_df[
    (debug_df["y_true"] == 1) &
    (debug_df["missed_by_n_models"] > 0)
].copy()
diagnostic_cols = [
    "y_true",
    "log_reg_pred",
    "log_reg_proba",
    "random_forest_pred",
    "random_forest_proba",
    "gradient_boosting_pred",
    "gradient_boosting_proba",
    "Process temperature [K]",
    "Temperature difference",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Power [W]",
    "Tool wear [min]",
    "Type_L",
    "Type_M"
]

print(
    hard_cases[diagnostic_cols + ["missed_by_n_models"]]
    .sort_values("missed_by_n_models", ascending=False)
)

index: {4151, 4833, 4475}

In [ ]:
df_raw = pd.read_csv('../data/raw/produkcja.csv')
df_raw['Power [W]'] = df_raw['Rotational speed [rpm]'] * df_raw['Torque [Nm]'] * (2 * np.pi / 60)
df_raw['Temperature difference'] = df_raw['Process temperature [K]'] - df_raw['Air temperature [K]']
df_raw['Kryterium_OSF'] = df_raw['Tool wear [min]'] * df_raw['Torque [Nm]']

In [ ]:
fn_every = [3787, 442, 3760, 3829, 1085, 2494, 9653, 4151, 4833, 2941, 9663, 8026, 4475, 9758, 7510, 9018, 4034, 8609, 8199]

In [ ]:
suspect_cases = df_raw.loc[fn_every].copy()
print(suspect_cases)

442 - bardzo blisko dolnego krańca przedziału dla [PWF]

3760, 3787, 3829, 4151, 4833, 4475 - bardzo blisko kranców przedziału dla obu parametrów - rotational speed i temp diff [HDF]

1085, 2494, 9653, 9663, 8026 - algorytmy nie wiedziały o Kryterium_OSF [OSF]

2941, 9758 - wysoki Rotational speed oraz Tool wear [TWF] - w EDA nie znaleziono zależności między wysokim Rotational speed a TWF

7510, 9018, 4034, 8609, 8199 - wysoki Tool wear [TWF]


3787, 442, 3760, 3829, 1085, 2494, 9653, 4151 - jeden model pomylił się

4833, 2941, 9663, 8026, 4475, 9758 - dwa modele pomyliły się

7510, 9018, 4034, 8609, 8199 - wszystkie się pomyliły

In [ ]:
mechanism_groups = {
    "PWF_boundary_case": [442],
    
    "HDF_boundary_case": [3760, 3787, 3829, 4151, 4833, 4475],
    
    "OSF_missing_engineered_criterion": [1085, 2494, 9653, 9663, 8026],
    
    "TWF_high_rotational_speed_and_tool_wear": [2941, 9758],
    
    "TWF_high_tool_wear": [7510, 9018, 4034, 8609, 8199]
}

In [ ]:
print('debug_df:', debug_df.shape)
print("Random Forest:", type(random_forest).__name__)
print("Gradient Boosting:", type(gradient_boosting).__name__)

## Analiza fałszywych awarii - False Positives

In [ ]:
fp_rf = debug_df[(debug_df['y_true']==0) & (debug_df['random_forest_pred']== 1)].copy()

fp_gb = debug_df[(debug_df['y_true']==0) & (debug_df['gradient_boosting_pred']== 1)].copy()

print("False positives - Random Forest:", len(fp_rf))
print("False positives - Gradient Boosting:", len(fp_gb))

In [ ]:
print(fp_rf)
print(fp_gb)


In [ ]:
print("RF — wartości y_true:")
print(fp_rf["y_true"].value_counts())

print("\nRF — wartości predykcji:")
print(fp_rf["random_forest_pred"].value_counts())

print("\nGB — wartości y_true:")
print(fp_gb["y_true"].value_counts())

print("\nGB — wartości predykcji:")
print(fp_gb["gradient_boosting_pred"].value_counts())

In [ ]:
fp_rf_summary = fp_rf.drop(columns=["log_reg_pred", "log_reg_proba", "gradient_boosting_pred", "gradient_boosting_proba", "missed_by_log_reg", "missed_by_gradient_boosting", "missed_by_n_models", "missed_by_random_forest"]).copy()
print(fp_rf_summary)

False positives dla Random Forest: 3764
4898
8198
7081
7677
4115
4110
7255
4862

In [ ]:
print(fp_rf_summary["random_forest_proba"].min())
print(fp_rf_summary["random_forest_proba"].max())

Przedział Random Forest Proba = [0.53; 0.75]

In [ ]:
fp_gb_summary = fp_gb.drop(columns=["log_reg_pred", "log_reg_proba", "random_forest_pred", "random_forest_proba", "missed_by_log_reg", "missed_by_gradient_boosting", "missed_by_n_models", "missed_by_random_forest"]).copy()
print(fp_gb_summary)

False positives dla Gradient Boosting: 6925 3764 7677 1507 7935 4862 9671

In [ ]:
print(fp_gb_summary["gradient_boosting_proba"].min().round(2))
print(fp_gb_summary["gradient_boosting_proba"].max().round(2))

Przedział Gradient Boosting Proba = (0.54; 0.85)

In [ ]:
common_fp = fp_rf_summary.merge(fp_gb_summary, left_index=True, right_index=True, suffixes=('_rf', '_gb'))
print(common_fp)

In [ ]:
common_fp[['random_forest_proba', 'gradient_boosting_proba']].round(3)

In [ ]:

fp_rf_summary['random_forest_margin'] = (fp_rf_summary['random_forest_proba'] - DECISION_THRESHOLD)

fp_gb_summary['gradient_boosting_margin'] = (fp_gb_summary['gradient_boosting_proba'] - DECISION_THRESHOLD)

In [ ]:
print(fp_rf_summary[['random_forest_proba', 'random_forest_margin']].round(3))

In [ ]:
print(fp_gb_summary[['gradient_boosting_proba', 'gradient_boosting_margin']].round(3))

In [ ]:
osf_threshold = 11003.2

fp_rf_summary['OSF_distance'] = (fp_rf_summary['Torque [Nm]'] * fp_rf_summary['Tool wear [min]']) - osf_threshold
fp_gb_summary['OSF_distance'] = (fp_gb_summary['Torque [Nm]'] * fp_gb_summary['Tool wear [min]']) - osf_threshold

fp_rf_summary['OSF_like'] = fp_rf_summary['OSF_distance'] >= 0
fp_gb_summary['OSF_like'] = fp_gb_summary['OSF_distance'] >= 0


In [ ]:
fp_rf_summary['HDF_like'] = fp_rf_summary['Temperature difference'].between(7.6, 8.6) & fp_rf_summary['Rotational speed [rpm]'].between(1212, 1379)
fp_gb_summary['HDF_like'] = fp_gb_summary['Temperature difference'].between(7.6, 8.6) & fp_gb_summary['Rotational speed [rpm]'].between(1212, 1379)


In [ ]:
fp_rf_summary['PWF_like'] = fp_rf_summary['PWF_like'] = (
    ((fp_rf_summary['Power [W]'] > 1148) & (fp_rf_summary['Power [W]'] < 3477))
    |
    ((fp_rf_summary['Power [W]'] > 9004) & (fp_rf_summary['Power [W]'] < 10469))
)
fp_gb_summary['PWF_like'] = fp_gb_summary['PWF_like'] = (
    ((fp_gb_summary['Power [W]'] > 1148) & (fp_gb_summary['Power [W]'] < 3477))
    |
    ((fp_gb_summary['Power [W]'] > 9004) & (fp_gb_summary['Power [W]'] < 10469))
)


In [ ]:
fp_rf_summary['TWF_like'] = fp_rf_summary['Tool wear [min]'] > 198
fp_gb_summary['TWF_like'] = fp_gb_summary['Tool wear [min]'] > 198

In [ ]:
print(fp_rf_summary.head(15))

In [ ]:
print(fp_gb_summary.head(15))

In [ ]:
print(f"RF OSF: {fp_rf_summary['OSF_like'].sum()}")
print(f"RF HDF: {fp_rf_summary['HDF_like'].sum()}")
print(f"RF PWF: {fp_rf_summary['PWF_like'].sum()}")
print(f"RF TWF: {fp_rf_summary['TWF_like'].sum()}")
print(f"GB OSF: {fp_gb_summary['OSF_like'].sum()}")
print(f"GB HDF: {fp_gb_summary['HDF_like'].sum()}")
print(f"GB PWF: {fp_gb_summary['PWF_like'].sum()}")
print(f"GB TWF: {fp_gb_summary['TWF_like'].sum()}")

In [ ]:
cols_like = ['OSF_like', 'HDF_like', 'PWF_like', 'TWF_like']

fp_rf_summary[cols_like].sum(axis=1)

In [ ]:
fp_gb_summary[cols_like].sum(axis=1)

## Zapis wyników eksperymentu bazowego

In [ ]:
from pathlib import Path

# 1. Utworzenie folderu na wyniki
results_path = Path("../results")
results_path.mkdir(parents=True, exist_ok=True)

# 2. Zapis metryk modeli
results_df.round(6).to_csv(
    results_path / "model_results_baseline.csv",
    index=False
)

# 3. Przygotowanie i zapis false negatives
false_negatives_export = (
    hard_cases[
        diagnostic_cols + ["missed_by_n_models"]
    ]
    .sort_values(
        by="missed_by_n_models",
        ascending=False
    )
    .copy()
)
proba_columns = [
    "log_reg_proba",
    "random_forest_proba",
    "gradient_boosting_proba"
]

false_negatives_export[proba_columns] = (
    false_negatives_export[proba_columns].round(3)
)
false_negatives_export.to_csv(
    results_path / "false_negatives_baseline.csv",
    index=True,
    index_label="record_index"
)

# 4. Przygotowanie false positives
fp_rf_export = fp_rf_summary.rename(
    columns={"Kryterium_OSF": "OSF_distance"}
).copy()

fp_gb_export = fp_gb_summary.rename(
    columns={"Kryterium_OSF": "OSF_distance"}
).copy()
for df in [fp_rf_export, fp_gb_export]:

    df["OSF_distance"] = (
        df["Tool wear [min]"] * df["Torque [Nm]"] - 11003.2
    )

    df["OSF_like"] = df["OSF_distance"] >= 0

    df["HDF_like"] = (
        df["Temperature difference"].between(7.6, 8.6)
        & df["Rotational speed [rpm]"].between(1212, 1379)
    )

    df["PWF_like"] = (
        df["Power [W]"].between(1148.44, 3477.24)
        | df["Power [W]"].between(9004.43, 10469.92)
    )

    df["TWF_like"] = df["Tool wear [min]"].between(198, 253)
like_columns = [
    "OSF_like",
    "HDF_like",
    "PWF_like",
    "TWF_like"
]

fp_rf_export["n_like_mechanisms"] = (
    fp_rf_export[like_columns].sum(axis=1)
)

fp_gb_export["n_like_mechanisms"] = (
    fp_gb_export[like_columns].sum(axis=1)
)

fp_rf_export["model"] = "Random Forest"
fp_gb_export["model"] = "Gradient Boosting"

fp_rf_export = fp_rf_export.rename(
    columns={
        "random_forest_proba": "predicted_probability",
        "random_forest_margin": "margin"
    }
)

fp_gb_export = fp_gb_export.rename(
    columns={
        "gradient_boosting_proba": "predicted_probability",
        "gradient_boosting_margin": "margin"
    }
)

false_positive_columns = [
    "model",
    "predicted_probability",
    "margin",
    "Process temperature [K]",
    "Temperature difference",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Power [W]",
    "Tool wear [min]",
    "OSF_distance",
    "OSF_like",
    "HDF_like",
    "PWF_like",
    "TWF_like",
    "n_like_mechanisms"
]

false_positives_export = pd.concat(
    [
        fp_rf_export[false_positive_columns],
        fp_gb_export[false_positive_columns]
    ],
    axis=0
)

false_positives_export = (
    false_positives_export
    .reset_index()
    .rename(columns={"index": "record_index"})
    .sort_values(
        by=["model", "predicted_probability"],
        ascending=[True, False]
    )
)
columns_to_round = [
    "predicted_probability",
    "margin",
    "OSF_distance"
]

false_positives_export[columns_to_round] = (
    false_positives_export[columns_to_round].round(3)
)
false_positives_export.to_csv(
    results_path / "false_positives_baseline.csv",
    index=False
)

print("Zapisano pliki:")
print(results_path / "model_results_baseline.csv")
print(results_path / "false_negatives_baseline.csv")
print(results_path / "false_positives_baseline.csv")